# Snowball v6 — US + TW 2,000-Stock Purged Walk-Forward Training (Kaggle GPU/CPU)

Train and benchmark Snowball's multi-factor equity models on **2,000 combined stocks** (**US 1,000 + TW 1,000** across 13 standardized industries) with strict anti-look-ahead controls:

- **Zero Label Overlap (Purged Expanding Walk-Forward)**: every refit strictly uses `date <= refit_date - horizon_weeks`.
- **Next-Trading-Day Open Execution**: forward returns run from Monday open (`t+1`) to Monday open (`t+1+H`), USD-normalized via historical `TWD=X`.
- **4-Model OOS Comparison**:
  1. `baseline_v6_prior` — Zero-parameter macro-conditioned prior
  2. `stage1_icir_shrinkage` — Bayesian Shrinkage ICIR rolling weights
  3. `stage2_ridge` — Cross-sectional $L_2$-regularized Ridge regression
  4. `stage3_lgbm` — Shallow heavily-regularized LightGBM (`max_depth=3`, `num_leaves=7`, `reg_lambda=25`)
- **Portable Production Export**: writes `/kaggle/working/snowball_artifacts/v6_learned_weights.json` (loadable directly into `global_ranker.py --learned-weights`) + `lgbm_model.txt` + `oos_summary.json`.

> **Kaggle Checklist**: Turn on **Internet** (if building panel directly on Kaggle) and optionally select **GPU T4 x2** (optional; LightGBM auto-falls back to multi-core CPU). Click **Save Version -> Save & Run All** so `/kaggle/working/snowball_artifacts/` is persisted as Output.

In [ ]:
import os, sys, subprocess, platform
from pathlib import Path

REPO_URL = "https://github.com/948787878787/snowball.git"
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()

# Locate or clone Snowball repo
local_candidates = [
    Path.cwd().parent / "skills" / "snowball" / "scripts",
    Path.cwd() / "skills" / "snowball" / "scripts",
    WORK_DIR / "snowball" / "skills" / "snowball" / "scripts",
]
SCRIPTS_DIR = next((p for p in local_candidates if p.exists()), None)
if SCRIPTS_DIR is None:
    repo_dir = WORK_DIR / "snowball"
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    SCRIPTS_DIR = repo_dir / "skills" / "snowball" / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# Ensure yfinance is available on Kaggle
try:
    import yfinance  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])

print("Environment :", "Kaggle" if IS_KAGGLE else platform.platform())
print("Scripts Dir :", SCRIPTS_DIR)
print("CPU Cores   :", os.cpu_count())

In [ ]:
# === Training Configuration (US 1,000 + TW 1,000 = 2,000 Combined Stocks) ===
MARKETS = ("US", "TW")
HISTORY_PERIOD = "8y"          # Recommended: 8y-10y for multi-regime walk-forward
HORIZON_WEEKS = 4               # 4-week forward excess-return rank target
MIN_TRAIN_WEEKS = 104           # 2-year minimum expanding warmup before first OOS prediction
REFIT_STEP_WEEKS = 8            # Refit every 8 weeks (approx. bi-monthly)
RIDGE_ALPHA = 500.0             # Strong L2 shrinkage for noisy equity cross-sections
TOP_K_PORTFOLIO = 40            # Top-40 combined US+TW long-only weekly TWR simulation
USE_GPU = False                 # Set True if Kaggle GPU T4/P100 is attached
OFFLINE_DEMO = False            # Set True for fast <15s synthetic sanity check without network

# Optional: path to Point-in-Time historical fundamentals CSV (columns: ticker, available_date, rule40_pct, ...)
PIT_FUNDAMENTALS_CSV = None

ARTIFACT_DIR = WORK_DIR / "snowball_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Artifacts will be exported to:", ARTIFACT_DIR)

## 1. Load Pre-Uploaded Kaggle Dataset OR Build Live 2,000-Stock Panel
If you uploaded `snowball_us_tw_2000.parquet` as a Private Kaggle Dataset under `/kaggle/input/`, this cell loads it immediately. Otherwise, it builds the 2,000-stock Point-in-Time panel live and caches it to `/kaggle/working/snowball_us_tw_2000.parquet`.

In [ ]:
import pandas as pd
from build_training_panel import build_combined_panel, load_dataframe, save_dataframe

cached_candidates = []
if Path("/kaggle/input").exists():
    cached_candidates.extend(sorted(Path("/kaggle/input").rglob("snowball_us_tw_2000.*")))
cached_candidates.append(WORK_DIR / "snowball_us_tw_2000.parquet")
cached_candidates.append(WORK_DIR / "snowball_us_tw_2000.csv.gz")

existing_panel = next((p for p in cached_candidates if p.exists() and not OFFLINE_DEMO), None)

if existing_panel is not None:
    print(f"Loading pre-built training panel: {existing_panel}")
    panel = load_dataframe(existing_panel)
else:
    print(f"Building combined {MARKETS} panel ({'OFFLINE DEMO' if OFFLINE_DEMO else HISTORY_PERIOD + ' LIVE'})...")
    panel = build_combined_panel(
        markets=MARKETS,
        period=HISTORY_PERIOD,
        horizons=(HORIZON_WEEKS, 12),
        offline_demo=OFFLINE_DEMO,
        demo_stocks_per_market=150 if OFFLINE_DEMO else None,
        pit_fundamentals_csv=Path(PIT_FUNDAMENTALS_CSV) if PIT_FUNDAMENTALS_CSV else None,
        batch_size=150,
    )
    saved = save_dataframe(panel, WORK_DIR / "snowball_us_tw_2000.parquet")
    print(f"Cached panel for reuse: {saved}")

print(f"\nPanel Shape : {panel.shape}")
print(f"Date Range  : {panel['date'].min().date()} -> {panel['date'].max().date()} ({panel['date'].nunique()} weeks)")
print("Tickers/Mkt :", panel.groupby("market")["ticker"].nunique().to_dict())
panel.head(3)

## 2. Run Strict Purged Expanding Walk-Forward Training
Trains and evaluates `baseline_v6_prior`, `stage1_icir_shrinkage`, `stage2_ridge`, and `stage3_lgbm` on identical out-of-sample (OOS) dates, enforcing `max_train_date <= refit_date - HORIZON_WEEKS`.

In [ ]:
from train_walkforward import (
    run_purged_walkforward,
    export_training_artifacts,
    format_metrics_table,
)

eff_min_train = min(MIN_TRAIN_WEEKS, max(36, panel["date"].nunique() // 3))

result = run_purged_walkforward(
    panel=panel,
    horizon=HORIZON_WEEKS,
    min_train_weeks=eff_min_train,
    refit_step_weeks=REFIT_STEP_WEEKS,
    ridge_alpha=RIDGE_ALPHA,
    enable_lgbm=True,
    use_gpu=USE_GPU,
    top_k=TOP_K_PORTFOLIO,
)

exported = export_training_artifacts(result, ARTIFACT_DIR)

metrics_df = format_metrics_table(result["metrics"])
print("=== Out-of-Sample Model Comparison (Combined US + TW) ===")
display(metrics_df.round(4))

print("\nLearned v6 6-Pillar Weights (ready for global_ranker.py --learned-weights):")
print(result["learned_v6_weights"])

print("\nPurge Gap Audit (first & last refit):")
print("  First:", result["purge_audit"][0])
print("  Last :", result["purge_audit"][-1])

## 3. OOS Diagnostics & Visualizations
Inspect cumulative OOS Rank IC, rolling Stage-1 ICIR pillar weights, and LightGBM feature importance.

In [ ]:
import matplotlib.pyplot as plt
from train_walkforward import _compute_daily_ic_series

oos_df = result["oos_predictions"]
target_col = f"target_rank_{HORIZON_WEEKS}w"
model_names = list(result["metrics"].keys())

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

# Panel 1: Cumulative OOS Rank IC
for m in model_names:
    ic_s = _compute_daily_ic_series(oos_df, f"pred_{m}", target_col)
    axes[0].plot(ic_s.index, ic_s.cumsum(), label=f"{m} (IC={ic_s.mean():.3f})", linewidth=2)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_title("Cumulative OOS Rank IC (US + TW)")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.25)

# Panel 2: Rolling Stage-1 ICIR Pillar Weights
wh = result["weight_history"].copy()
wh["refit_date"] = pd.to_datetime(wh["refit_date"])
w_cols = [c for c in wh.columns if c.startswith("w_")]
for c in w_cols:
    axes[1].plot(wh["refit_date"], wh[c], label=c.replace("w_", ""), linewidth=2)
axes[1].set_title("Stage 1 Bayesian Shrinkage Pillar Weights")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)

# Panel 3: LightGBM Feature Importance (if trained)
imp = result["lgbm_feature_importance"]
if imp:
    imp_s = pd.Series(imp).sort_values(ascending=True)
    axes[2].barh(imp_s.index, imp_s.values, color="#3b82f6")
    axes[2].set_title("Stage 3 LightGBM Feature Importance")
else:
    axes[2].text(0.5, 0.5, "LightGBM not installed", ha="center", va="center")
    axes[2].set_title("Stage 3 LightGBM Feature Importance")
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "oos_diagnostics.png", dpi=150)
plt.show()
print("Exported files:", exported)

## 4. Deploy Back to Local Snowball Ranker
Download `/kaggle/working/snowball_artifacts/v6_learned_weights.json` from this notebook's **Output** tab and run locally:

```bash
python skills/snowball/scripts/global_ranker.py \
  --market GLOBAL --model v6 \
  --learned-weights ./v6_learned_weights.json --top 20
```